In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset, DatasetDict
from transformers import T5Tokenizer, T5ForConditionalGeneration, TrainingArguments, Trainer, DataCollatorForSeq2Seq
import torch
import wandb
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def prepare_grouped_dataset(jsonl_path, train_size=0.8, val_size=0.1):
    df = pd.read_json(jsonl_path, lines=True)

    # Mapeo senior: IA -> positive, Humano -> negative
    df['target_text'] = df['is_real'].map({0: "positive", 1: "negative"})
    df['input_text'] = "classify authenticity: " + df['title']

    # 1. Separar Test set (Hold-out)
    gs_test = GroupShuffleSplit(n_splits=1, train_size=train_size + val_size, random_state=42)
    train_val_idx, test_idx = next(gs_test.split(df, groups=df['group_id']))

    df_train_val = df.iloc[train_val_idx]
    df_test = df.iloc[test_idx]

    # 2. Separar Train y Eval del resto
    relative_train_size = train_size / (train_size + val_size)
    gs_eval = GroupShuffleSplit(n_splits=1, train_size=relative_train_size, random_state=42)
    train_idx, eval_idx = next(gs_eval.split(df_train_val, groups=df_train_val['group_id']))

    df_train = df_train_val.iloc[train_idx]
    df_eval = df_train_val.iloc[eval_idx]

    ds_dict = DatasetDict({
        'train': Dataset.from_pandas(df_train),
        'validation': Dataset.from_pandas(df_eval),
        'test': Dataset.from_pandas(df_test)
    })

    return ds_dict


In [ ]:
model_name = "google-t5/t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)

ds_dict = prepare_grouped_dataset("/content/drive/MyDrive/TFG/titles_data.jsonl")

def tokenize_fn(examples):
    inputs = tokenizer(examples["input_text"], max_length=512, truncation=True, padding="max_length")
    targets = tokenizer(text_target=examples["target_text"], max_length=10, truncation=True, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = ds_dict.map(tokenize_fn, batched=True)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # En T5, los logits suelen ser una tupla, tomamos la primera parte
    predictions = np.argmax(logits[0], axis=-1)

    pos_label_id = tokenizer.encode("positive")[0]
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels.flatten(), predictions.flatten(), average='binary', pos_label=pos_label_id, zero_division=0
    )

    return {
        'accuracy': accuracy_score(labels.flatten(), predictions.flatten()),
        'f1_positive': f1,
        'precision_positive': precision,
        'recall_positive': recall
    }

def get_sentinel_prediction(text, model, tokenizer):
    input_ids = tokenizer("classify authenticity: " + text, return_tensors="pt").input_ids.to("cuda")

    with torch.no_grad():
        outputs = model(input_ids=input_ids, decoder_input_ids=torch.tensor([[tokenizer.pad_token_id]]).to("cuda"))
        logits = outputs.logits[:, 0, :]

    pos_id = tokenizer.encode("positive")[0]
    neg_id = tokenizer.encode("negative")[0]

    probs = torch.softmax(logits[:, [pos_id, neg_id]], dim=-1)
    return "IA" if probs[0][0] > probs[0][1] else "Humano"

def evaluate_model(model, tokenizer, dataset):
    predictions = []
    true_labels = []

    for example in dataset:
        predicted_label = get_sentinel_prediction(example['input_text'].replace('classify authenticity: ', ''), model, tokenizer)
        predictions.append(predicted_label)
        true_labels.append(example['target_text'])

    mapped_predictions = ["positive" if p == "IA" else "negative" for p in predictions]

    print("\n--- Resultados de Evaluación ---")
    accuracy = accuracy_score(true_labels, mapped_predictions)
    print(f"Accuracy: {accuracy:.4f}")

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, mapped_predictions, average=None, labels=['positive', 'negative'], zero_division=0
    )

    print("\nMétricas por clase:")
    print(f"  Clase 'positive' (IA):\n    Precision: {precision[0]:.4f}\n    Recall:    {recall[0]:.4f}\n    F1-score:  {f1[0]:.4f}")
    print(f"  Clase 'negative' (Humano):\n    Precision: {precision[1]:.4f}\n    Recall:    {recall[1]:.4f}\n    F1-score:  {f1[1]:.4f}")

    return true_labels, mapped_predictions

def plot_confusion_matrix(y_true, y_pred):
    labels = ['positive', 'negative']
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['IA (Synthetic)', 'Humano (Real)'],
                yticklabels=['IA (Synthetic)', 'Humano (Real)'])

    plt.title('Matriz de Confusión: T5 Sentinel - Titulares', fontsize=14)
    plt.xlabel('Predicción del Modelo', fontsize=12)
    plt.ylabel('Etiqueta Real', fontsize=12)

    plt.savefig('matriz_confusion_t5.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
sweep_config = {
    'method': 'bayes',
    'metric': {
        'name': 'eval/f1_positive',
        'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 1e-6,
            'max': 1e-4
        },
        'batch_size': {
            'values': [8, 16, 32]
        },
        'grad_acc': {
            'values': [2, 4, 8]
        },
        'epochs': {
            'value': 5
        },
        'weight_decay': {
            'values': [0.0, 0.01, 0.1]
        }
    }
}

sweep_id = wandb.sweep(sweep_config, project="TFG-T5-Detector")


In [ ]:
def train_sweep():
    with wandb.init() as run:
        config = wandb.config

        # Instantiate fresh model for each run
        model = T5ForConditionalGeneration.from_pretrained(model_name)
        data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

        training_args = TrainingArguments(
            output_dir="./temp_results",
            learning_rate=config.learning_rate,
            per_device_train_batch_size=config.batch_size,
            gradient_accumulation_steps=config.grad_acc,
            num_train_epochs=config.epochs,
            weight_decay=config.weight_decay,
            eval_strategy="epoch",
            report_to="wandb",
            fp16=True
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset["train"],
            eval_dataset=tokenized_dataset["validation"],
            data_collator=data_collator,
            compute_metrics=compute_metrics
        )

        trainer.train()


In [ ]:
# Ejecutar el sweep (ajusta count según los recursos de Colab que tengas disponibles)
wandb.agent(sweep_id, function=train_sweep, count=10)


In [ ]:
# Sustituir con los mejores valores del Sweep tras ejecutarlo
best_config = {
    "lr": 4.5e-5,
    "batch_size": 8,
    "grad_acc": 8,
    "weight_decay": 0.01
}

print("Iniciando entrenamiento final con la mejor configuración...")

final_model = T5ForConditionalGeneration.from_pretrained(model_name)
final_data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=final_model)

final_args = TrainingArguments(
    output_dir="./model_final_tfg",
    num_train_epochs=5,
    learning_rate=best_config["lr"],
    per_device_train_batch_size=best_config["batch_size"],
    gradient_accumulation_steps=best_config["grad_acc"],
    weight_decay=best_config["weight_decay"],
    fp16=True,
    report_to="none"
)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=final_data_collator
)

final_trainer.train()

print("\n--- EVALUACIÓN FINAL EN CONJUNTO DE TEST (UNSEEN DATA) ---")
true_labels, mapped_predictions = evaluate_model(final_model, tokenizer, tokenized_dataset["test"])
plot_confusion_matrix(true_labels, mapped_predictions)
